In [1]:
import pandas as pd
import numpy as np

In [ ]:
# load datasets
data = pd.read_csv('../data/raw/ab_test.csv')
country = pd.read_csv('../data/raw/countries_ab.csv')

In [ ]:
# display the first 5 rows
data.head()

,id,time,con_treat,page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [ ]:
# display the first 5 rows
country.head()

,id,country
0,834778,UK
1,928468,US
2,822059,UK
3,711597,UK
4,710616,UK


In [ ]:
# check data dataset information
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         294478 non-null  int64 
 1   time       294478 non-null  object
 2   con_treat  294478 non-null  object
 3   page       294478 non-null  object
 4   converted  294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [ ]:
# check country dataset information
country.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290584 entries, 0 to 290583
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   id       290584 non-null  int64 
 1   country  290584 non-null  object
dtypes: int64(1), object(1)
memory usage: 4.4+ MB


## check duplicate records

In [ ]:
# check duplicate on id column
data['id'].duplicated().sum()

np.int64(3894)

In [ ]:
# remove duplicate records
data = data.drop_duplicates(subset='id', keep='first')

In [ ]:
# make sure no duplicate records
data['id'].duplicated().sum()

np.int64(0)

## cleaning ab_test data

In [ ]:
# check unique values in the con_treat column
data['con_treat'].unique()

array(['control', 'treatment'], dtype=object)

In [ ]:
# check unique values in the page column
data['page'].unique()

array(['old_page', 'new_page'], dtype=object)

control -> old_page

treatment -> new_page

In [ ]:
# Check whether the control group is assigned only to the old page and save the result to con_data
con_data = data[data['con_treat'] == 'control']
con_data['page'].unique()

array(['old_page', 'new_page'], dtype=object)

In [ ]:
# Check whether the treatment group is assigned only to the new page and save the result to treat_data
treat_data = data[data['con_treat'] == 'treatment']
treat_data['page'].unique()

array(['new_page', 'old_page'], dtype=object)

In [ ]:
# Identify users in the control group who were assigned the new page
con_data[con_data['page'] == 'new_page']

,id,time,con_treat,page,converted
22,767017,58:15.0,control,new_page,0
240,733976,11:16.4,control,new_page,0
490,808613,44:01.3,control,new_page,0
846,637639,09:52.7,control,new_page,1
850,793580,25:33.7,control,new_page,1
...,...,...,...,...,...
277083,731502,54:36.5,control,new_page,0
277283,904541,19:25.2,control,new_page,0
281176,828478,45:27.8,control,new_page,0
281593,917949,16:02.2,control,new_page,1


In [ ]:
# Identify users in the treatment group who were assigned the old page
treat_data[treat_data['page'] == 'old_page']

,id,time,con_treat,page,converted
308,857184,34:59.8,treatment,old_page,0
327,686623,26:40.7,treatment,old_page,0
357,856078,29:30.4,treatment,old_page,0
685,666385,11:54.8,treatment,old_page,0
713,748761,47:44.4,treatment,old_page,0
...,...,...,...,...,...
293773,688144,34:50.5,treatment,old_page,1
293817,876037,15:09.0,treatment,old_page,1
293917,738357,37:55.7,treatment,old_page,0
294014,813406,25:33.2,treatment,old_page,0


- There are 1,928 mismatched records in the control group where users were assigned the new_page.
- There are 1,965 mismatched records in the treatment group where users were assigned the old_page.

In [ ]:
# Correct page assignments by replacing new_page with old_page for control group users
con_data['page'] = con_data['page'].replace('new_page', 'old_page')

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21852\1307736545.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  con_data['page'] = con_data['page'].replace('new_page', 'old_page')


In [ ]:
# Verify that all users in the control group are assigned to old_page
con_data['page'].unique()

array(['old_page'], dtype=object)

In [ ]:
# Correct page assignments by replacing old_page with new_page for treatment group users
treat_data['page'] = treat_data['page'].replace('old_page', 'new_page')

# Verify that all users in the treatment group are assigned to new_page
treat_data['page'].unique()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21852\4181149970.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  treat_data['page'] = treat_data['page'].replace('old_page', 'new_page')


array(['new_page'], dtype=object)

In [ ]:
# Combine the cleaned control and treatment groups into a single dataset
data_new = pd.concat([con_data, treat_data], ignore_index=True)

In [23]:
data_new.head()

,id,time,con_treat,page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,864975,52:26.2,control,old_page,1
3,936923,20:49.1,control,old_page,0
4,719014,48:29.5,control,old_page,0


In [24]:
data_new[(data_new['con_treat'] == 'control') & (data_new['page'] == 'new_page')]

,id,time,con_treat,page,converted


In [25]:
data_new[(data_new['con_treat'] == 'treatment') & (data_new['page'] == 'old_page')]

,id,time,con_treat,page,converted


## cleaning countreis_ab

In [ ]:
# display the first 5 rows
country.head()

,id,country
0,834778,UK
1,928468,US
2,822059,UK
3,711597,UK
4,710616,UK


In [ ]:
# check dataset information
country.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290584 entries, 0 to 290583
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   id       290584 non-null  int64 
 1   country  290584 non-null  object
dtypes: int64(1), object(1)
memory usage: 4.4+ MB


In [ ]:
# check duplicate data
country.duplicated().sum()

np.int64(0)

In [ ]:
# check null data
country.isnull().sum()

id         0
country    0
dtype: int64

## merge country dan data_new

In [ ]:
# merge country dataset and data_new
data_new = pd.merge(data_new, country, on='id', how='left')

In [35]:
data_new.head()

,id,time,con_treat,page,converted,country
0,851104,11:48.6,control,old_page,0,US
1,804228,01:45.2,control,old_page,0,US
2,864975,52:26.2,control,old_page,1,US
3,936923,20:49.1,control,old_page,0,US
4,719014,48:29.5,control,old_page,0,US


In [ ]:
# save data to csv
data_new.to_csv('../data/cleaned/ab_test_cleaned.csv', index=False)